In [1]:
import os
import requests
import pandas as pd
from alive_progress import alive_bar
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()
print("Finished imports.")
google_maps_api = os.getenv("GOOGLE_MAPS_API")

Finished imports.


In [2]:
# Load the CSV file containing scored edge data, including geometry
edges_df = pd.read_csv('../../routing/all_scored_edges_filtered_with_ai.csv')
a = pd.read_csv('../output/accessibility_scored_edges_slope.csv')

C:\Users\vzhang\AppData\Local\Temp\ipykernel_20292\607858699.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  edges_df = pd.read_csv('../../routing/all_scored_edges_filtered_with_ai.csv')


In [3]:
def get_elevation(lat, lon):
    """
    Fetch elevation data for a given latitude and longitude using Google Maps Elevation API.

    Parameters:
        lat (float): Latitude.
        lon (float): Longitude.
        api_key (str): Google Maps API key with Elevation API enabled.

    Returns:
        float: Elevation in meters.
    
    Raises:
        Exception: If API request fails or response is invalid.
    """
    url = "https://maps.googleapis.com/maps/api/elevation/json"
    params = {
        "locations": f"{lat},{lon}",
        "key": google_maps_api
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        raise Exception(f"Error fetching elevation: {response.status_code}")

    data = response.json()
    if data['status'] != 'OK' or not data['results']:
        raise Exception(f"Error in API response: {data.get('status')}")

    elevation = data['results'][0]['elevation']
    return elevation


In [4]:
def parse_linestring(wkt_str):
    """
    Parses a WKT LINESTRING or MULTILINESTRING and returns a flat list of (lat, lon) tuples.

    Parameters:
        wkt_str (str): A WKT string in LINESTRING or MULTILINESTRING format.

    Returns:
        List[Tuple[float, float]]: List of (lat, lon) tuples.
    """
    wkt_str = wkt_str.strip()
    
    if wkt_str.startswith("LINESTRING"):
        coord_part = wkt_str[len("LINESTRING"):].strip(" ()")
        segments = [coord_part]
        
    elif wkt_str.startswith("MULTILINESTRING"):
        coord_part = wkt_str[len("MULTILINESTRING"):].strip(" ()")
        segments = coord_part.split("), (")
        
    else:
        raise ValueError("Input must start with 'LINESTRING' or 'MULTILINESTRING'")

    latlon_list = []

    for segment in segments:
        coords = segment.replace("(", "").replace(")", "").split(",")
        for pair in coords:
            lon_str, lat_str = pair.strip().split()
            latlon_list.append((float(lat_str), float(lon_str)))

    return latlon_list

In [5]:
elevations = []  # Will hold a list of elevation lists per edge

# Create a progress bar to show elevation-fetching progress
with alive_bar(len(edges_df), force_tty=True) as bar:
    try:
        for index, row in edges_df.iterrows():
            row_el = []  # List to hold elevations for one edge
            point_list = parse_linestring(row["geometry"])  # Convert geometry string to list of (lon, lat) tuples

            # Get elevation for each coordinate in the linestring
            for tup in point_list:
                elevation = get_elevation(tup[0], tup[1])  # Call elevation API or lookup function
                row_el.append(elevation)  # Append elevation to this edge's list

            elevations.append(row_el)  # Store the list of elevations for this edge
            bar()  # Update the progress bar

    except Exception as e:
        # Handle any errors by printing what went wrong and saving partial progress
        print("Partial elevations:", elevations)
        raise e


|████████████████████████████████████████| 18972/18972 [100%] in 54:17.5 (5.82/s ▇▇▅ 2474/18972 [13%] in 7:26 (~49:30, ▆▄▂ 4612/18972 [24%] in 13:48 (~42:50 ▃▁▃ 4651/18972 [25%] in 13:55 (~42:50 ▄▂▂ 4658/18972 [25%] in 13:56 (~42:50 ▂▄▆ 4749/18972 [25%] in 14:12 (~42:30 ▃▁▃ 6632/18972 [35%] in 19:48 (~36:50 ▁▃▅ 8544/18972 [45%] in 25:33 (~31:10 ▂▂▄ 9696/18972 [51%] in 28:34 (~27:20 ▆█▆ 10372/18972 [55%] in 30:28 (~25:1 ▅▇▇ 10823/18972 [57%] in 31:48 (~23:5 ▇▇▅ 13205/18972 [70%] in 38:07 (~16:3 ▃▅▇ 13469/18972 [71%] in 38:51 (~15:5 ▅▇▇ 13683/18972 [72%] in 39:37 (~15:1 ▅▃▁ 14159/18972 [75%] in 40:57 (~13:5 █▆▄ 16000/18972 [84%] in 46:00 (~8:30 ▃▁▃ 17627/18972 [93%] in 50:30 (~3:50 ▆█▆ 18532/18972 [98%] in 52:57 (~1:10


In [6]:
edges_df['elevation_m_lmsl'] = elevations
a['elevation_m_lmsl'] = edges_df['elevation_m_lmsl'].fillna(0)
edges_df.to_csv('../../routing/all_scored_edges_filtered_with_ai.csv', index=False)
a.to_csv('../output/accessibility_scored_edges_slope.csv', index=False)